In [3]:
%load_ext dockermagic

The dockermagic extension is already loaded. To reload it, use:
  %reload_ext dockermagic


In [60]:
%%dockerwrite hadoop /opt/script.sql

USE bikeshare;
drop table if exists status_updates;
drop table if exists profiles;
drop table if exists gender_summary;
drop table if exists school_summary;

-- show tables
SHOW TABLES;

Successfully copied 2.05kB to hadoop:/opt/script.sql


In [61]:
%%dockerexec hadoop

source /opt/envvars.sh

beeline -n hadoop -u jdbc:hive2://localhost:10000 --silent=true -f /opt/script.sql









+------------+
|  tab_name  |
+------------+
| stations   |
| stations2  |
| trips      |
| trips2     |
+------------+


exemplo artigo (facebook)

In [66]:
%%dockerwrite hadoop /opt/script.sql

USE bikeshare;
drop table if exists status_updates;
CREATE TABLE status_updates (
  userid INT,
  status STRING
)
PARTITIONED BY (ds STRING)
STORED AS ORC;

INSERT INTO status_updates 
PARTITION(ds='2025-01-01') VALUES (0, 'feliz'), (2, 'triste'), (6, 'surpreso'), (8, 'triste'), (9, 'surpreso');

INSERT INTO status_updates 
PARTITION(ds='2025-01-02')  VALUES (3, 'triste'), (4, 'feliz'), (0, 'triste'); 

INSERT INTO status_updates 
PARTITION(ds='2025-01-03') VALUES (1, 'surpreso'), (5, 'triste'), (7, 'surpreso'), (0, 'feliz');

drop table if exists profiles;
CREATE TABLE profiles (
  userid INT,
  school STRING,
  gender INT
)
STORED AS ORC;

insert into profiles (userid, school, gender) values
 (0, 'unicamp', 0),
 (1, 'ufrgs', 1),
 (2, 'ufrgs', 0),
 (3, 'ufrgs', 1),
 (4, 'unicamp', 1),
 (5, 'unicamp', 0),
 (6, 'ufrgs', 0),
 (7, 'unicamp', 1),
 (8, 'puc', 0),
 (9, 'puc', 1);


-- show tables
SHOW TABLES;

DESCRIBE FORMATTED status_updates;
DESCRIBE FORMATTED profiles;

Successfully copied 2.56kB to hadoop:/opt/script.sql


In [67]:
%%dockerexec hadoop

source /opt/envvars.sh

beeline -n hadoop -u jdbc:hive2://localhost:10000 --silent=true -f /opt/script.sql











 





 
























+-----------------+
|    tab_name     |
+-----------------+
| profiles        |
| stations        |
| stations2       |
| status_updates  |
| trips           |
| trips2          |
+-----------------+


+-------------------------------+----------------------------------------------------+-----------------------------+
|           col_name            |                     data_type                      |           comment           |
+-------------------------------+----------------------------------------------------+-----------------------------+
| # col_name                    | data_type                                          | comment                     |
| userid                        | int                                                |                             |
| status                        | string                                             |                             |
|                               | NULL               

In [68]:
%%dockerwrite hadoop /opt/script.sql

USE bikeshare;

select * from status_updates;

select * from profiles;


Successfully copied 2.05kB to hadoop:/opt/script.sql


In [69]:
%%dockerexec hadoop

source /opt/envvars.sh

beeline -n hadoop -u jdbc:hive2://localhost:10000 --silent=true -f /opt/script.sql




+------------------------+------------------------+--------------------+
| status_updates.userid  | status_updates.status  | status_updates.ds  |
+------------------------+------------------------+--------------------+
| 0                      | feliz                  | 2025-01-01         |
| 2                      | triste                 | 2025-01-01         |
| 6                      | surpreso               | 2025-01-01         |
| 8                      | triste                 | 2025-01-01         |
| 9                      | surpreso               | 2025-01-01         |
| 3                      | triste                 | 2025-01-02         |
| 4                      | feliz                  | 2025-01-02         |
| 0                      | triste                 | 2025-01-02         |
| 1                      | surpreso               | 2025-01-03         |
| 5                      | triste                 | 2025-01-03         |
| 7                      | surpreso             

multi-table insert

In [70]:
%%dockerwrite hadoop /opt/script.sql

USE bikeshare;

drop table if exists gender_summary;
CREATE TABLE gender_summary (
  gender INT,
  count INT
)
PARTITIONED BY (ds STRING)
STORED AS ORC;

drop table if exists school_summary;
CREATE TABLE school_summary (
  school STRING,
  count INT
)
PARTITIONED BY (ds STRING)
STORED AS ORC;

FROM (SELECT a.status, b.school, b.gender
      FROM status_updates a JOIN profiles b
        ON (a.userid = b.userid 
       and a.ds='2025-01-01')
) subq1
INSERT OVERWRITE TABLE gender_summary
  PARTITION(ds='2025-01-01')
  SELECT subq1.gender, COUNT(1) GROUP BY subq1.gender
INSERT OVERWRITE TABLE school_summary
  PARTITION(ds='2025-01-01')
  SELECT subq1.school, COUNT(1) GROUP BY subq1.school;

Successfully copied 2.56kB to hadoop:/opt/script.sql


In [71]:
%%dockerexec hadoop

source /opt/envvars.sh

beeline -n hadoop -u jdbc:hive2://localhost:10000 --silent=true -f /opt/script.sql

In [72]:
%%dockerwrite hadoop /opt/script.sql

USE bikeshare;

select * from gender_summary;

select * from school_summary;

Successfully copied 2.05kB to hadoop:/opt/script.sql


In [73]:
%%dockerexec hadoop

source /opt/envvars.sh

beeline -n hadoop -u jdbc:hive2://localhost:10000 --silent=true -f /opt/script.sql




+------------------------+-----------------------+--------------------+
| gender_summary.gender  | gender_summary.count  | gender_summary.ds  |
+------------------------+-----------------------+--------------------+
| 0                      | 4                     | 2025-01-01         |
| 1                      | 1                     | 2025-01-01         |
+------------------------+-----------------------+--------------------+


+------------------------+-----------------------+--------------------+
| school_summary.school  | school_summary.count  | school_summary.ds  |
+------------------------+-----------------------+--------------------+
| puc                    | 2                     | 2025-01-01         |
| ufrgs                  | 2                     | 2025-01-01         |
| unicamp                | 1                     | 2025-01-01         |
+------------------------+-----------------------+--------------------+


fazendo novo insert na partição, aumenta nro de arquivos?

In [51]:
%%dockerwrite hadoop /opt/script.sql

USE bikeshare;

INSERT INTO status_updates 
PARTITION(ds='2025-01-01') VALUES (10, 'feliz'), (11, 'triste');

Successfully copied 2.05kB to hadoop:/opt/script.sql


In [52]:
%%dockerexec hadoop

source /opt/envvars.sh

beeline -n hadoop -u jdbc:hive2://localhost:10000 --silent=true -f /opt/script.sql

insert overwrite

In [54]:
%%dockerwrite hadoop /opt/script.sql

USE bikeshare;

select * from status_updates 
where ds='2025-01-03';

Successfully copied 2.05kB to hadoop:/opt/script.sql


In [55]:
%%dockerexec hadoop

source /opt/envvars.sh

beeline -n hadoop -u jdbc:hive2://localhost:10000 --silent=true -f /opt/script.sql





+------------------------+------------------------+--------------------+
| status_updates.userid  | status_updates.status  | status_updates.ds  |
+------------------------+------------------------+--------------------+
| 1                      | surpreso               | 2025-01-03         |
| 5                      | triste                 | 2025-01-03         |
| 7                      | surpreso               | 2025-01-03         |
+------------------------+------------------------+--------------------+


In [58]:
%%dockerwrite hadoop /opt/script.sql

USE bikeshare;

INSERT OVERWRITE TABLE status_updates 
PARTITION(ds='2025-01-03') VALUES 
    (100,'feliz'),
    (500,'surpreso'),
    (700,'triste');

select * from status_updates 
where ds='2025-01-03';

Successfully copied 2.05kB to hadoop:/opt/script.sql


In [59]:
%%dockerexec hadoop

source /opt/envvars.sh

beeline -n hadoop -u jdbc:hive2://localhost:10000 --silent=true -f /opt/script.sql











+------------------------+------------------------+--------------------+
| status_updates.userid  | status_updates.status  | status_updates.ds  |
+------------------------+------------------------+--------------------+
| 100                    | feliz                  | 2025-01-03         |
| 500                    | surpreso               | 2025-01-03         |
| 700                    | triste                 | 2025-01-03         |
+------------------------+------------------------+--------------------+
